# 02 Bronze ADLS Ingestion - Healthcare

Loads 3 CSV feeds from the volume into Bronze Delta tables using Auto Loader.

| Source | Bronze table |
|---|---|
| `patient_encounters/*.csv` | `bronze_patient_encounters` |
| `event_data/lab_test_events*.csv` | `bronze_lab_test_events` |
| `event_data/bed_transfer_events*.csv` | `bronze_bed_transfer_events` |

In [0]:
CATALOG = "capstone_8hospital"
SCHEMA = "bronze_schema"
VOLUME = "ext_capstone8_vol"

SCHEMA_NAME = f"{CATALOG}.{SCHEMA}"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
STATE_ROOT = f"{VOLUME_ROOT}/_autoloader_state/bronze"

In [0]:
from pyspark.sql import functions as F

dataset_name = "patient_encounters"
source_path = f"{VOLUME_ROOT}/patient_encounters"
file_pattern = "*.csv"
target_table = f"{SCHEMA_NAME}.bronze_patient_encounters"

schema_path = f"{STATE_ROOT}/{dataset_name}/schema"
checkpoint_path = f"{STATE_ROOT}/{dataset_name}/checkpoint"

# Auto Loader reads CSV files incrementally.
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .option("pathGlobFilter", file_pattern)
    .load(source_path)
)

# Add ingestion metadata. No business transformations are performed.
bronze_df = (
    source_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# Append to the Bronze Delta table and retain stream progress.
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

# Wait until ingestion finishes.
query.awaitTermination()

print(f"PASS | Completed ingestion for {dataset_name}")
print(f"      Target table: {target_table}")

In [0]:
dataset_name = "lab_test_events"
source_path = f"{VOLUME_ROOT}/event_data"
file_pattern = "lab_test_events*.csv"
target_table = f"{SCHEMA_NAME}.bronze_lab_test_events"

schema_path = f"{STATE_ROOT}/{dataset_name}/schema"
checkpoint_path = f"{STATE_ROOT}/{dataset_name}/checkpoint"

# Auto Loader reads CSV files incrementally.
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .option("pathGlobFilter", file_pattern)
    .load(source_path)
)

# Add ingestion metadata. No business transformations are performed.
bronze_df = (
    source_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# Append to the Bronze Delta table and retain stream progress.
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

# Wait until ingestion finishes.
query.awaitTermination()

print(f"PASS | Completed ingestion for {dataset_name}")
print(f"      Target table: {target_table}")

In [0]:
dataset_name = "bed_transfer_events"
source_path = f"{VOLUME_ROOT}/event_data"
file_pattern = "bed_transfer_events*.csv"
target_table = f"{SCHEMA_NAME}.bronze_bed_transfer_events"

schema_path = f"{STATE_ROOT}/{dataset_name}/schema"
checkpoint_path = f"{STATE_ROOT}/{dataset_name}/checkpoint"

# Auto Loader reads CSV files incrementally.
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .option("pathGlobFilter", file_pattern)
    .load(source_path)
)

# Add ingestion metadata. No business transformations are performed.
bronze_df = (
    source_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# Append to the Bronze Delta table and retain stream progress.
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

# Wait until ingestion finishes.
query.awaitTermination()

print(f"PASS | Completed ingestion for {dataset_name}")
print(f"      Target table: {target_table}")

In [0]:
results = []

for name in ["patient_encounters", "lab_test_events", "bed_transfer_events"]:
    table_name = f"{SCHEMA_NAME}.bronze_{name}"
    results.append((table_name, spark.table(table_name).count()))

display(spark.createDataFrame(results, ["table_name", "row_count"]))

In [0]:
display(
    spark.table(f"{SCHEMA_NAME}.bronze_patient_encounters")
    .limit(10)
)
